# Phase 2 Equations

\begin{equation}
Q_r(s,a_r) \gets \mathbb{E}_g \mathbb{E}_{a_H \sim \pi_H(s,g)} \mathbb{E}_{s' \sim s,a} \gamma_r V_r(s')
\tag{4}
\end{equation}

\begin{equation}
\pi_r(s)(a_r) \propto (-Q_r(s,a_r))^{-\beta_r}
\tag{5}
\end{equation}

\begin{equation}
V_h^e(s,g_h) \gets \mathbb{E}_{a_r \sim \pi_r(s)} \mathbb{E}_{g_{-h}} \mathbb{E}_{a_H \sim \pi_H(s,g)} \mathbb{E}_{s' \sim s,a} [U_h(s',g_h) + \gamma_h V_h^e(s', g_h)]
\tag{6}
\end{equation}

\begin{equation}
X_h(s) \gets \sum_{g_h \in \mathcal{G}_h} V_h^e(s,g_h)^\zeta
\tag{7}
\end{equation}

\begin{equation}
U_r(s) \gets - (\sum_h X_h(s)^{-\xi})^\eta
\tag{8}
\end{equation}

\begin{equation}
V_r(s) \gets U_r(s) + \mathbb{E}_{a_r \sim \pi_r(s)} Q_r(s,a_r)
\tag{9}
\end{equation}

# Special Case for goal independent human policy

If we assume $\pi_H(s,g) = \pi_H(s)$ then (4) and (6) simplify.

\begin{equation}
Q_r(s,a_r) \gets \mathbb{E}_{a_H \sim \pi_H(s)} \mathbb{E}_{s' \sim s,a} \gamma_r V_r(s')
\tag{4*}
\end{equation}

\begin{equation}
V_h^e(s,g_h) \gets \mathbb{E}_{a_r \sim \pi_r(s)} \mathbb{E}_{a_H \sim \pi_H(s)} \mathbb{E}_{s' \sim s,a} [U_h(s',g_h) + \gamma_h V_h^e(s', g_h)]
\tag{6*}
\end{equation}

# Robots Move first

In [10]:
from src.world_model.grid_world_core import GridWorldCore, Pos, GridWorldState, Actions
from src.world_model.grid_world_env import GridWorldEnv, _ALL_DIRECTIONS
from src.world_model.solver import EmpoParameter, BackwardInductionSolver

# params
width = 3
height = 1
max_steps = 5
# H E R
#   ↑
human_pos = Pos(0, 0)
robot_pos = Pos(2, 0)
goal = Pos(1, 0)
mexican_standoff = GridWorldState(
    crates=(),
    robots=(robot_pos,),
    humans=(human_pos,),
    time_step=0,
)


# environment
gw_mexican = GridWorldCore(
    width=width,
    height=height,
    walls=frozenset(),
    max_steps=max_steps,
    goals=((goal,),),
)

# robot moves first
gw_mexican.step(
    mexican_standoff,
    robot_actions=(Actions.LEFT,),
    human_actions=(Actions.RIGHT,),
)

GridWorldState(crates=(), robots=(Pos(x=1, y=0),), humans=(Pos(x=0, y=0),), time_step=1)

# Code Example

In [3]:
from src.world_model.grid_world_core import GridWorldCore, Pos, GridWorldState
from src.world_model.grid_world_env import GridWorldEnv, _ALL_DIRECTIONS
from src.world_model.solver import EmpoParameter, BackwardInductionSolver

# params
# H R E
#   ↑
human_pos = Pos(0, 0)
robot_pos = Pos(1, 0)
goal = Pos(1, 0)
please_move = GridWorldState(
    crates=(),
    robots=(robot_pos,),
    humans=(human_pos,),
    time_step=0,
)

# environment
gw_please_move = GridWorldCore(
    width=width,
    height=height,
    walls=frozenset(),
    max_steps=max_steps,
    goals=((goal,),),
)

# distribution environment & solver setup
solver_params = EmpoParameter(
    gamma_r=1,
    beta_r=1,
    gamma_h=1,
    zeta=2,
    xi=1,
    eta=1,
)

distr_env_please_move = GridWorldEnv(gw_please_move)
solver = BackwardInductionSolver(distr_env_please_move, solver_params)

# solving
robot_policy = solver.compute_robot_policy(please_move)
Q_r = solver.Q_r

In [4]:
for action in _ALL_DIRECTIONS:
    print(
        action,
        robot_policy[please_move, action],
        Q_r[please_move, action],
    )

Actions.STAY 0.17173285651055298 -71.86622521289809
Actions.UP 0.17173285651055298 -71.86622521289809
Actions.DOWN 0.17173285651055298 -71.86622521289809
Actions.LEFT 0.17173285651055298 -71.86622521289809
Actions.RIGHT 0.31306857395778814 -39.42200900722085


In [5]:
please_move_now = GridWorldState(
    crates=(),
    robots=(robot_pos,),
    humans=(human_pos,),
    time_step=max_steps - 2,
)

for action in _ALL_DIRECTIONS:
    print(
        action,
        robot_policy[please_move_now, action],
        Q_r[please_move_now, action],
    )

Actions.STAY 0.04332441775391791 -594.8849976204635
Actions.UP 0.04332441775391791 -594.8849976204635
Actions.DOWN 0.04332441775391791 -594.8849976204635
Actions.LEFT 0.04332441775391791 -594.8849976204635
Actions.RIGHT 0.8267023289843284 -31.175727040846404


In [9]:
# solving mexican_standoff
distr_env_mexican = GridWorldEnv(gw_mexican)
solver_mexican = BackwardInductionSolver(distr_env_mexican, solver_params)

# solving
policy_mexican = solver_mexican.compute_robot_policy(mexican_standoff)
Q_mexican = solver_mexican.Q_r

for action in _ALL_DIRECTIONS:
    print(
        action,
        policy_mexican[mexican_standoff, action],
        Q_mexican[mexican_standoff, action],
    )

Actions.STAY 0.21985042241042344 -39.42200900722085
Actions.UP 0.21985042241042344 -39.42200900722085
Actions.DOWN 0.21985042241042344 -39.42200900722085
Actions.LEFT 0.1205983103583063 -71.86622521289809
Actions.RIGHT 0.21985042241042344 -39.42200900722085


# Questions

### 1. Naming

- Name for Phase II?
- What does $m$ and $e$ stand for in $V_h^m$ and $V_h^e$?

### 2. Softmax

Robot policy softmax?

$$
\pi_r(s,a) \propto e^{-\beta \ln(-Q(s,a))}
$$

vs

$$
\pi_r(s,a) \propto e^{-Q(s,a)/\tau}

### 3. Goals

#### Changing goals?

>  Humans hae many possible and potentially changing goals $g_h \in \mathcal{G}_h$

How are changing goals modeled?

#### States in goals mutually unreachable

> We require that the desirable states in a goal $g_h$ are mutually unreachable.

$s, s' \in g_h \Rightarrow \neg\exists \tau \; s,s' \in \tau$

Why?

# Insights

- give humans real goals, maybe changing after they have reached them
- robot conciders all possible goals
- possible goals need a prior (uniformative we don't want to model humanities goals)
- $V_h$ talks about likelihood
- policy is a power law policy

